<h1><strong><center> Pipeline </center></strong></h1>


Этот скрипт выполняет предсказания на основе предварительно обученных моделей.  
Чтобы запустить его, следуйте инструкциям:

1. **Укажите пути к моделям в словаре `model_paths`**  
2. **Загрузите тестовые данные из файла**  
3. **Запустите скрипт и получите предсказания в файле `predictions.csv`**  

## 1. Загрузка библиотек и настройка окружения
Здесь загружаются необходимые библиотеки, включая Pandas, Pickle для работы с моделями и CatBoost для градиентного бустинга.

In [41]:
import pandas as pd
import pickle
from itertools import combinations_with_replacement
from sklearn.base import BaseEstimator, ClassifierMixin
from catboost import CatBoostClassifier
import warnings

warnings.filterwarnings('ignore')

In [53]:
class BlendingPipeline(BaseEstimator, ClassifierMixin):
    def __init__(self, model_paths):
        '''Инициализация класса'''
        self.model_paths = model_paths
        self.models = {}
        self.meta_model = None
    
    def load_models(self):
        '''Загрузка моделей из файлов'''
        self.models['gradient_boosting'] = pickle.load(open(self.model_paths['gradient_boosting'], 'rb'))
        self.models['logistic_regression'] = pickle.load(open(self.model_paths['logistic_regression'], 'rb'))
        self.models['random_forest'] = pickle.load(open(self.model_paths['random_forest'], 'rb'))
        self.models['extra_trees'] = pickle.load(open(self.model_paths['extra_trees'], 'rb'))
        self.models['catboost'] = CatBoostClassifier()
        self.models['catboost'].load_model(self.model_paths['catboost'])
        self.meta_model = pickle.load(open(self.model_paths['meta_model'], 'rb'))
    
    def feature_engineering(self, data):
        '''Генерация новых признаков из исходных данных
        
        Параметры:
              data (pd.DataFrame) - исходные данные
        
        Возвращает:
             pd.DataFrame - обработанные данные с новыми признаками
        '''
        for col1, col2 in combinations_with_replacement(data.columns, 2):  # combinations_with_replacement для включения самих себя
            # Операции только для разных признаков
            if col1 != col2:  # Для суммы и произведения — без самоповторений
                data[f'{col1}_plus_{col2}'] = data[col1] + data[col2]
                data[f'{col1}_proizvedenie_{col2}'] = data[col1] * data[col2]
            
            # Для разности — с самоповторами тоже
            data[f'{col1}_minus_{col2}'] = data[col1] - data[col2]
        
        return data
    
    def predict(self, X):
        '''Предсказание мета-модели
        
        Параметры:
              X (pd.DataFrame) - входные данные для предсказания
        
        Возвращает:
             np.array - предсказанные значения
        '''
        X = self.feature_engineering(X)
        meta_features = self._get_meta_features(X)
        return self.meta_model.predict(meta_features)
    
    def _get_meta_features(self, X):
        '''Формирование мета-признаков из предсказаний базовых моделей
        
        Параметры:
              X (pd.DataFrame) - входные данные
        
        Возвращает:
             pd.DataFrame - мета-признаки
        '''
        meta_features = pd.DataFrame()
        
        model_names = {
            'gradient_boosting': 'gb',
            'logistic_regression': 'lr',
            'random_forest': 'rf',
            'extra_trees': 'et',
            'catboost': 'catboost'
        }
        
        for full_name, short_name in model_names.items():
            meta_features[short_name] = self.models[full_name].predict(X)
        
        correct_order = ['catboost', 'lr', 'rf', 'et', 'gb']
        meta_features = meta_features[correct_order]
        
        return meta_features

# Укажите пути к моделям  

Перед запуском убедитесь, что все пути к файлам моделей корректны.  
Если структура папок отличается, обновите словарь `model_paths`:

- `gradient_boosting`: путь к файлу модели градиентного бустинга (pkl)  
- `logistic_regression`: путь к файлу модели логистической регрессии (pkl)  
- `random_forest`: путь к файлу случайного леса (pkl)  
- `extra_trees`: путь к файлу Extra Trees (pkl)  
- `catboost`: путь к модели CatBoost (cb)  
- `meta_model`: путь к мета-модели (pkl)

In [54]:
model_paths = {
    'gradient_boosting': '../../models/3_final_models/optuna_gradient_boosting_model.pkl',
    'logistic_regression': '../../models/3_final_models/optuna_logistic_regression_model.pkl',
    'random_forest': '../../models/3_final_models/optuna_random_forest_model.pkl',
    'extra_trees': '../../models/3_final_models/et_model.pkl',
    'catboost': '../../models/3_final_models/catboost_model.cb',
    'meta_model': '../../models/3_final_models/meta_model_xgboost.pkl'
}

# создание и обучение пайплайна
blender = BlendingPipeline(model_paths)
blender.load_models()

# Загрузка тестовых данных  

Убедитесь, что файл `customer_data.csv` находится в указанной папке.  
Если файл в другом месте, обновите путь в коде.

ВНИМАНИЕ! 
Данные должны содержать только числовые признаки, все не числовые признаки будут УДАЛЕНЫ! 

In [55]:
# загрузка тестовых данных
X_test = pd.read_csv('../../data/raw_data/customer_data.csv').select_dtypes(include=['number']) # оставляем только числовые признаки (без ID пользователя)

# Очистка данных  

Перед предсказанием удаляются строки с пропущенными значениями.  
Если вы хотите обработать пропуски иначе (например, заполнить средними значениями),  
добавьте соответствующий код вместо `dropna()`.

In [56]:
# предсказание на данных без пропущенных значений
X_test_cleaned = X_test.dropna().copy()
predictions = blender.predict(X_test_cleaned)
predictions

array([2, 2, 2, ..., 2, 2, 2], dtype=int64)

# Сохранение предсказаний  

Результаты предсказаний сохраняются в CSV-файл.  
Укажите путь, куда сохранить результаты, вместо `predictions.csv`.

In [57]:
# сохранение результатов
pd.DataFrame(predictions, columns=['Prediction']).to_csv('predictions.csv', index=False)

,Prediction
0,2
1,2
2,2
3,2
4,2
...,...
8631,2
8632,2
8633,2
8634,2
